Starting with $u(x,y)=1-3x^2+2x^3$ again.  
But this time we will use the PDE he gave us,   
$-\nabla \cdot(K(x,y) \nabla u) + \sigma^2 u = f$  strong form  
aka $-\left[\frac{\partial}{\partial x}(K(x,y) \frac{\partial}{\partial x}u) + \frac{\partial}{\partial y}(K(x,y) \frac{\partial}{\partial y} u)\right] + \sigma^2 u = f$  
$\int_\Omega\left[\left(K(x,y) \frac{\partial}{\partial x} u \frac{\partial}{\partial x} \phi + K(x,y) \frac{\partial}{\partial y} u \frac{\partial}{\partial y} \phi\right) + \sigma^2 u \phi \right] dxdy= \int_\Omega f \phi dxdy$   weak form  
where $K(x,y)=1+x^2+y^2$ and $\sigma=\frac{1}{2}$.
    
Results: Doesn't appear to be matching

<span style="color:red">Current Concern: Direct Solve and CG Solve visually match, but don't match original function  
Suspect I'm coding the PDE incorrectly, or the method? 
Consider error calculations in addition to visual inspection.</span>


In [ ]:
# Import any packages needed
from ngsolve import *
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Domain is unit square, create initial mesh
N = 64 
mesh = Mesh(unit_square.GenerateMesh(maxh=1/N))
mesh.nv, mesh.ne
Draw(mesh);

In [ ]:
# Working with H1-conforming piecewise linear finite elements
# boundary conditions are Dirichlet on the left and default=Neumann on the other 3 sides
fes = H1(mesh, order=1, dirichlet="left")
fes.ndof

# Solution will be stored as a grid function
gfu = GridFunction(fes)
gfu.Set(1, BND) # this sets the Dirichlet boundary component to 1, might want to modify later
Draw(gfu)

In [ ]:
# Set test and trial spaces
u = fes.TrialFunction()
phi = fes.TestFunction()

# # Coefficient function for K(x,y)
Kxy = CoefficientFunction(1+x*x+y*y)
# #Kxy = CoefficientFunction(3)
sigma = 0.5

# Bilinear form
a = BilinearForm(fes)
a += Kxy*grad(u)*grad(phi)*dx+sigma*sigma*u*phi*dx
a.Assemble()

print(a.mat)

In [ ]:
# Right hand side
f = LinearForm(fes)
fakef = 25/4 -12*x + 12*x*y - 12*x*y*y + 69/4 *x*x - 12*x*x*y - 47/2 *x*x*x + 6*y*y
f += fakef*phi*dx
f.Assemble()

print(f.vec)

In [ ]:
# Solve the PDE
c = Preconditioner(a,"local")
c.Update()
solvers.BVP(bf=a, lf=f, gf=gfu, pre=c, maxsteps=500, tol=1e-14)
Draw(gfu)

In [ ]:
# Plot original function for comparison
plant = CoefficientFunction(1-3*x*x+2*x*x*x)
Draw(plant, mesh)